In [ ]:
import torch
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
import os


# 加载BLIP模型和处理器

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")


def load_image(image_path, resize=True):
    image = Image.open(image_path).convert("RGB")  # 确保是RGB
    if resize:
        image = image.resize((512, 512))
    return image


def describe_image_with_blip(image_path, num_descriptions=3):
    image = load_image(image_path)
    inputs = processor(images=image, return_tensors="pt")

    descriptions = []
    for _ in range(num_descriptions):
        out = model.generate(
            **inputs,
            max_length=50,               
            num_beams=5,                 
            no_repeat_ngram_size=2,      
            early_stopping=True,         
            do_sample=True,              
            top_k=50,
            top_p=0.95,
            temperature=1.0
        )
        description = processor.decode(out[0], skip_special_tokens=True)
        descriptions.append(description)

    # 选择最长的一条作为最详细描述
    best_description = max(descriptions, key=len)
    return best_description, descriptions


def describe_folder(folder_path):
    supported_exts = (".jpg", ".jpeg", ".png")
    image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(supported_exts)]

    for image_file in image_files:
        image_path = os.path.join(folder_path, image_file)
        best_desc, all_descs = describe_image_with_blip(image_path)
        print(f"\nImage: {image_file}")
        print("Generated detailed English description:\n", best_desc)
        print("\nAll generated descriptions:")
        for i, desc in enumerate(all_descs, 1):
            print(f"{i}. {desc}")


# 主函数
if __name__ == "__main__":
    folder_path = r"C:\Users\gidle\Desktop\sample"
    describe_folder(folder_path)



Image: 01274_mask.png
Generated detailed English description:
 a man sitting on a couch wearing a hat

All generated descriptions:
1. a man wearing a hat
2. a man wearing a hat and a jacket
3. a man sitting on a couch wearing a hat

Image: 01275_mask.png
Generated detailed English description:
 a man in a suit and tie speaking into microphones

All generated descriptions:
1. a man in a suit and tie speaking into microphones
2. a man in a suit speaking into microphones
3. a man in a suit speaking into microphones

Image: 01276_mask.png
Generated detailed English description:
 a little boy standing on the edge of a swimming pool

All generated descriptions:
1. a young boy is playing on a tram tram
2. a little boy standing on the edge of a swimming pool
3. a young boy standing in front of a tree

Image: 01284_mask.png
Generated detailed English description:
 a man sitting at a table with food in front of him

All generated descriptions:
1. a man sitting at a table with food in front of h

In [26]:
import pandas as pd
import os

csv_path = r"D:\Hateful-Image-Project\data\sample_data\top_800_info.csv"
image_folder = r"C:\Users\gidle\Desktop\sample"
output_csv = os.path.join(image_folder, "sample_info.csv") 

supported_exts = (".jpg", ".jpeg", ".png")
image_files = []
for f in os.listdir(image_folder):
    if f.lower().endswith(supported_exts):
        name = os.path.splitext(f)[0]  # 去掉扩展名
        name = name.replace('_mask', '')  # 去掉 _mask
        name = name.lstrip('0')           # 去掉前导零
        image_files.append(name)

df = pd.read_csv(csv_path)
# 处理 CSV 名称
df['img_clean'] = df['img'].str.split('/').str[-1]  # 去掉 img/
df['img_clean'] = df['img_clean'].str.replace('.png','',regex=False)  # 去掉扩展名
df['img_clean'] = df['img_clean'].str.replace('_mask','',regex=False) # 去掉 _mask
df['img_clean'] = df['img_clean'].str.lstrip('0')  # 去掉前导零

filtered_df = df[df['img_clean'].isin(image_files)]
filtered_df = filtered_df.drop(columns=['img_clean'])
filtered_df.to_csv(output_csv, index=False)

print(f"Filtered data saved to: {output_csv}, total rows: {len(filtered_df)}")


Filtered data saved to: C:\Users\gidle\Desktop\sample\sample_info.csv, total rows: 10


In [1]:
import torch
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
import os
import pandas as pd
import requests
import json
from tqdm import tqdm

API_SECRET_KEY = "sk-zk21884c49fc398427914f99fc30171ecb068f998dc6f05b"
ANALYSIS_URL = "https://api.zhizengzeng.com/v1/chat/completions"

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

def load_image(image_path, resize=True):
    image = Image.open(image_path).convert("RGB")
    if resize:
        image = image.resize((512, 512))
    return image

def describe_image_with_blip(image_path, num_descriptions=3):
    image = load_image(image_path)
    inputs = processor(images=image, return_tensors="pt")
    descriptions = []
    for _ in range(num_descriptions):
        out = model.generate(
            **inputs,
            max_length=50,
            num_beams=5,
            no_repeat_ngram_size=2,
            early_stopping=True,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=1.0
        )
        description = processor.decode(out[0], skip_special_tokens=True)
        descriptions.append(description)
    best_description = max(descriptions, key=len)
    return best_description, descriptions

def generate_analysis_prompt(description, csv_text):
    prompt = (
        f"请分析以下图像内容，结合提供的文本，联想之后，在BLIP生成的图像描述的基础上，生成一句客观且详细的英文的图像描述（不需要提到文本内容讲了什么），并提取5-10个关键词。\n"
        f"输出格式为JSON，字段：{{\"description\": \"...\", \"keywords\": [\"...\", \"...\"]}}\n"
        f"BLIP生成的图像描述：{description}\n"
        f"CSV中的文本内容：{csv_text}\n"
        f"注意：BLIP生成的描述可能不准确，可以结合文本思考。不要增加无关的信息，不要过度联想没有给你的图像信息（如我没有跟你说姿势，你描述这个人站着，这是不对的）。"
    )
    return prompt

def parse_json_from_llm(text):
    if text.startswith("```"):
        text = "\n".join(text.strip().split("\n")[1:-1])
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {"description": text, "keywords": []}

def analyze_caption_with_gpt4(description, csv_text):
    prompt = generate_analysis_prompt(description, csv_text)
    try:
        response = requests.post(
            ANALYSIS_URL,
            headers={
                'Content-Type': 'application/json',
                'Authorization': f'Bearer {API_SECRET_KEY}',
            },
            json={
                "model": "gpt-4.1",
                "messages": [{"role": "user", "content": prompt}]
            }
        )
        response.raise_for_status()
        result = response.json()
        if "choices" in result and len(result["choices"]) > 0:
            text = result["choices"][0]["message"]["content"]
            return parse_json_from_llm(text)
        else:
            return {"description": f"No choices returned: {result}", "keywords": []}
    except Exception as e:
        return {"description": f"Request failed: {e}", "keywords": []}

def clean_image_name(name):
    name = os.path.splitext(name)[0]
    name = name.replace('_mask', '')
    name = name.lstrip('0')
    return name

def process_folder_with_csv(folder_path, csv_path, output_csv="analysis_results.csv"):
    df = pd.read_csv(csv_path)
    df['img_clean'] = df['img'].str.split('/').str[-1]
    df['img_clean'] = df['img_clean'].str.replace('.png', '', regex=False)
    df['img_clean'] = df['img_clean'].str.replace('_mask', '', regex=False)
    df['img_clean'] = df['img_clean'].str.lstrip('0')

    supported_exts = (".jpg", ".jpeg", ".png")
    image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(supported_exts)]
    
    results = []

    for image_file in tqdm(image_files, desc="Processing images"):
        image_id = clean_image_name(image_file)
        matched_row = df[df['img_clean'] == image_id]
        csv_text = matched_row.iloc[0]['text'] if not matched_row.empty else ""

        image_path = os.path.join(folder_path, image_file)
        best_desc, all_descs = describe_image_with_blip(image_path)
        analysis_json = analyze_caption_with_gpt4(best_desc, csv_text)
        
        results.append({
            "image": image_file,
            "description": analysis_json.get("description", ""),
            "keywords": ", ".join(analysis_json.get("keywords", []))
        })

        print(f"\nImage: {image_file}")
        print("Best BLIP description:\n", best_desc)
        print("\nAll BLIP descriptions:")
        for i, desc in enumerate(all_descs, 1):
            print(f"{i}. {desc}")
        print("\nCSV text:\n", csv_text)
        print("\nAnalysis JSON:\n", analysis_json)

    df_results = pd.DataFrame(results)
    df_results.to_csv(output_csv, index=False)
    print(f"\nSaved structured analysis to {output_csv}")

if __name__ == "__main__":
    folder_path = r"C:\Users\gidle\Desktop\Inpainted"
    csv_path = r"C:\Users\gidle\Desktop\Inpainted\top_800_info.csv"
    process_folder_with_csv(folder_path, csv_path)


d:\Anaconda3\envs\blip_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Processing images:   0%|          | 1/800 [00:21<4:46:25, 21.51s/it]


Image: 01235_mask.png
Best BLIP description:
 a man in a black shirt and white hat with his hands up

All BLIP descriptions:
1. a man with a white hat and beard smiles at the camera
2. a man in a black shirt and a white hat
3. a man in a black shirt and white hat with his hands up

CSV text:
 when you're feeling horny asf but your habibi is on periods let's try a goat

Analysis JSON:
 {'description': 'A man wearing a black shirt and a white hat is raising his hands, and his facial expression suggests a playful or mischievous mood.', 'keywords': ['man', 'black shirt', 'white hat', 'hands up', 'mischievous', 'playful', 'adult', 'facial expression']}


Processing images:   0%|          | 2/800 [00:41<4:30:47, 20.36s/it]


Image: 01236_mask.png
Best BLIP description:
 a group of men standing in front of a sheep

All BLIP descriptions:
1. a group of men standing in front of a sheep
2. a group of men standing next to a sheep
3. two pictures of a man and a bunch of sheep

CSV text:
 meanwhile at the isis strip club

Analysis JSON:
 {'description': 'Several men are gathered together in front of a sheep, standing outdoors.', 'keywords': ['men', 'group', 'sheep', 'outdoors', 'gathered']}


Processing images:   0%|          | 3/800 [00:58<4:12:55, 19.04s/it]

...

Processing images: 100%|█████████▉| 799/800 [4:30:46<00:18, 18.42s/it]


Image: 07853_mask.png
Best BLIP description:
 a dog standing on a rock with a rainbow in the background

All BLIP descriptions:
1. a dog standing on a rock with a rainbow in the background
2. a dog standing on a rock with a rainbow in the background
3. a dog standing on top of a hill

CSV text:
 9/11 gets a mourning day while gay get a "pride" month because homosexuality is a bigger tragedy

Analysis JSON:
 {'description': 'A dog is on a rock with a rainbow seen in the sky behind, creating a vivid and colorful scene.', 'keywords': ['dog', 'rock', 'rainbow', 'sky', 'outdoors', 'colorful', 'vivid']}


Processing images: 100%|██████████| 800/800 [4:31:05<00:00, 20.33s/it]


Image: 07865_mask.png
Best BLIP description:
 a man and a woman are sitting on a bed

All BLIP descriptions:
1. a man and a woman are sitting on a bed
2. a man and woman sitting on a bed
3. a man and a woman laying on a bed

CSV text:
 a real man... loads the dishwasher every night!!

Analysis JSON:
 {'description': 'A man and a woman are sitting together on a bed, appearing to interact in a domestic setting.', 'keywords': ['man', 'woman', 'bed', 'sitting', 'domestic', 'interaction', 'together']}



Saved structured analysis to analysis_results.csv


## GPT 4.1

In [3]:
import torch
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
import os
import pandas as pd
import requests
import json

API_SECRET_KEY = "sk-zk21884c49fc398427914f99fc30171ecb068f998dc6f05b"
ANALYSIS_URL = "https://api.zhizengzeng.com/v1/chat/completions"

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

def load_image(image_path, resize=True):
    image = Image.open(image_path).convert("RGB")
    if resize:
        image = image.resize((512, 512))
    return image

def describe_image_with_blip(image_path, num_descriptions=3):
    image = load_image(image_path)
    inputs = processor(images=image, return_tensors="pt")

    descriptions = []
    for _ in range(num_descriptions):
        out = model.generate(
            **inputs,
            max_length=50,
            num_beams=5,
            no_repeat_ngram_size=2,
            early_stopping=True,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=1.0
        )
        description = processor.decode(out[0], skip_special_tokens=True)
        descriptions.append(description)

    best_description = max(descriptions, key=len)
    return best_description, descriptions

def generate_analysis_prompt(description, csv_text):
    prompt = (
        f"请分析以下图像内容，结合提供的文本，联想之后，在BLIP生成的图像描述的基础上，生成一句客观且详细的英文的图像描述（不需要提到文本内容讲了什么），并提取5-10个关键词。\n"
        f"输出格式为JSON，字段：{{\"description\": \"...\", \"keywords\": [\"...\", \"...\"]}}\n"
        f"BLIP生成的图像描述：{description}\n"
        f"CSV中的文本内容：{csv_text}\n"
        f"注意：BLIP生成的描述可能不准确，可以结合文本思考。不要增加无关的信息，不要过度联想没有给你的图像信息（如我没有跟你说姿势，你描述这个人站着，这是不对的）。"
    )
    return prompt

def parse_json_from_llm(text):
    """
    去掉可能存在的 Markdown 代码块并解析 JSON
    """
    # 去掉 ```json 或 ``` 包裹
    if text.startswith("```"):
        text = "\n".join(text.strip().split("\n")[1:-1])
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        # fallback
        return {"description": text, "keywords": []}

def analyze_caption_with_gpt4(description, csv_text):
    prompt = generate_analysis_prompt(description, csv_text)
    try:
        response = requests.post(
            ANALYSIS_URL,
            headers={
                'Content-Type': 'application/json',
                'Authorization': f'Bearer {API_SECRET_KEY}',
            },
            json={
                "model": "gpt-4.1",
                "messages": [{"role": "user", "content": prompt}]
            }
        )
        response.raise_for_status()
        result = response.json()
        if "choices" in result and len(result["choices"]) > 0:
            text = result["choices"][0]["message"]["content"]
            return parse_json_from_llm(text)
        else:
            return {"description": f"No choices returned: {result}", "keywords": []}
    except Exception as e:
        return {"description": f"Request failed: {e}", "keywords": []}

def clean_image_name(name):
    name = os.path.splitext(name)[0]
    name = name.replace('_mask', '')
    name = name.lstrip('0')
    return name

def process_folder_with_csv(folder_path, csv_path, output_csv="analysis_results.csv"):
    df = pd.read_csv(csv_path)
    df['img_clean'] = df['img'].str.split('/').str[-1]
    df['img_clean'] = df['img_clean'].str.replace('.png', '', regex=False)
    df['img_clean'] = df['img_clean'].str.replace('_mask', '', regex=False)
    df['img_clean'] = df['img_clean'].str.lstrip('0')

    supported_exts = (".jpg", ".jpeg", ".png")
    image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(supported_exts)]
    
    results = []

    for image_file in image_files:
        image_id = clean_image_name(image_file)
        matched_row = df[df['img_clean'] == image_id]
        csv_text = matched_row.iloc[0]['text'] if not matched_row.empty else ""

        image_path = os.path.join(folder_path, image_file)
        best_desc, all_descs = describe_image_with_blip(image_path)
        analysis_json = analyze_caption_with_gpt4(best_desc, csv_text)
        
        results.append({
            "image": image_file,
            "description": analysis_json.get("description", ""),
            "keywords": ", ".join(analysis_json.get("keywords", []))
        })

        print(f"\nImage: {image_file}")
        print("Best BLIP description:\n", best_desc)
        print("\nAll BLIP descriptions:")
        for i, desc in enumerate(all_descs, 1):
            print(f"{i}. {desc}")
        print("\nCSV text:\n", csv_text)
        print("\nAnalysis JSON:\n", analysis_json)

    # 保存结果
    df_results = pd.DataFrame(results)
    df_results.to_csv(output_csv, index=False)
    print(f"\nSaved structured analysis to {output_csv}")

if __name__ == "__main__":
    folder_path = r"C:\Users\gidle\Desktop\sample"
    csv_path = r"C:\Users\gidle\Desktop\sample\sample_info.csv"
    process_folder_with_csv(folder_path, csv_path)



Image: 01274_mask.png
Best BLIP description:
 a man sitting on a couch with his arms crossed

All BLIP descriptions:
1. a man wearing a hat
2. a man in a black jacket and a white hat
3. a man sitting on a couch with his arms crossed

CSV text:
 how you guys think you looks like but when pados wale pandit jee sees you taba renso

Analysis JSON:
 {'description': 'A man is sitting on a couch with his arms crossed, appearing thoughtful or possibly skeptical.', 'keywords': ['man', 'couch', 'sitting', 'arms crossed', 'thoughtful', 'skeptical']}

Image: 01275_mask.png
Best BLIP description:
 a man standing in front of a bunch of microphones

All BLIP descriptions:
1. a man in a suit and tie giving a speech
2. a man in a suit speaking into microphones
3. a man standing in front of a bunch of microphones

CSV text:
 if you think t willl stop eating palony and viannas voetsek!!! stupid no brain here

Analysis JSON:
 {'description': 'A man is speaking in front of several microphones, suggesting 

## Deepseek

In [27]:
import torch
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
import os
import pandas as pd
import requests
import json

API_KEY = "sk-db5bdb0ff3fb4f07981c5e0dbb3777b0"
API_URL = "https://api.deepseek.com/chat/completions"

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

def load_image(image_path, resize=True):
    image = Image.open(image_path).convert("RGB")
    if resize:
        image = image.resize((512, 512))
    return image

def describe_image_with_blip(image_path, num_descriptions=3):
    image = load_image(image_path)
    inputs = processor(images=image, return_tensors="pt")

    descriptions = []
    for _ in range(num_descriptions):
        out = model.generate(
            **inputs,
            max_length=50,
            num_beams=5,
            no_repeat_ngram_size=2,
            early_stopping=True,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=1.0
        )
        description = processor.decode(out[0], skip_special_tokens=True)
        descriptions.append(description)

    best_description = max(descriptions, key=len)
    return best_description, descriptions

def generate_deepseek_prompt(description, csv_text):
    prompt = (
        "You are a visual content analysis AI. Based on the BLIP-generated image description and the provided text from CSV, "
        "generate a single objective and detailed English description of the image. "
        "Also extract 5-10 relevant keywords representing objects, attributes, or scene elements.\n"
        "Output format must be JSON:\n"
        '{"description": "...", "keywords": ["...", "..."]}\n'
        "BLIP-generated image description: " + description + "\n"
        "CSV text content: " + csv_text + "\n"
        "Important instructions:\n"
        "1. Only describe what is visually present; do not infer unmentioned actions or poses.\n"
        "2. Avoid including information not supported by the image.\n"
        "3. Keywords should be concise and represent clear visual elements."
    )
    return prompt

def parse_json_from_llm(text):
    """
    去掉可能存在的 Markdown 代码块并解析 JSON
    """
    if text.startswith("```"):
        text = "\n".join(text.strip().split("\n")[1:-1])
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {"description": text, "keywords": []}

def analyze_caption_with_deepseek(description, csv_text):
    prompt = generate_analysis_prompt(description, csv_text)
    try:
        response = requests.post(
            API_URL,
            headers={
                'Content-Type': 'application/json',
                'Authorization': f'Bearer {API_KEY}',
            },
            json={
                "model": "deepseek-chat",  
                "messages": [{"role": "user", "content": prompt}]
            }
        )
        response.raise_for_status()
        result = response.json()
        if "choices" in result and len(result["choices"]) > 0:
            text = result["choices"][0]["message"]["content"]
            return parse_json_from_llm(text)
        else:
            return {"description": f"No choices returned: {result}", "keywords": []}
    except Exception as e:
        return {"description": f"Request failed: {e}", "keywords": []}

def clean_image_name(name):
    name = os.path.splitext(name)[0]
    name = name.replace('_mask', '')
    name = name.lstrip('0')
    return name

def process_folder_with_csv(folder_path, csv_path, output_csv="analysis_results_deepseek.csv"):
    df = pd.read_csv(csv_path)
    df['img_clean'] = df['img'].str.split('/').str[-1]
    df['img_clean'] = df['img_clean'].str.replace('.png', '', regex=False)
    df['img_clean'] = df['img_clean'].str.replace('_mask', '', regex=False)
    df['img_clean'] = df['img_clean'].str.lstrip('0')

    supported_exts = (".jpg", ".jpeg", ".png")
    image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(supported_exts)]
    
    results = []

    for image_file in image_files:
        image_id = clean_image_name(image_file)
        matched_row = df[df['img_clean'] == image_id]
        csv_text = matched_row.iloc[0]['text'] if not matched_row.empty else ""

        image_path = os.path.join(folder_path, image_file)
        best_desc, all_descs = describe_image_with_blip(image_path)
        analysis_json = analyze_caption_with_deepseek(best_desc, csv_text)
        
        results.append({
            "image": image_file,
            "description": analysis_json.get("description", ""),
            "keywords": ", ".join(analysis_json.get("keywords", []))
        })

        print(f"\nImage: {image_file}")
        print("Best BLIP description:\n", best_desc)
        print("\nAll BLIP descriptions:")
        for i, desc in enumerate(all_descs, 1):
            print(f"{i}. {desc}")
        print("\nCSV text:\n", csv_text)
        print("\nAnalysis JSON:\n", analysis_json)

    df_results = pd.DataFrame(results)
    df_results.to_csv(output_csv, index=False)
    print(f"\nSaved structured analysis to {output_csv}")

if __name__ == "__main__":
    folder_path = r"C:\Users\gidle\Desktop\sample"
    csv_path = r"C:\Users\gidle\Desktop\sample\sample_info.csv"
    process_folder_with_csv(folder_path, csv_path)



Image: 01274_mask.png
Best BLIP description:
 a man wearing a hat and jacket

All BLIP descriptions:
1. a man in a black jacket
2. a man wearing a hat
3. a man wearing a hat and jacket

CSV text:
 how you guys think you looks like but when pados wale pandit jee sees you taba renso

Analysis JSON:
 {'description': 'A man wearing a hat and a jacket, possibly in a casual or outdoor setting.', 'keywords': ['man', 'hat', 'jacket', 'outdoor', 'casual']}

Image: 01275_mask.png
Best BLIP description:
 a man in a suit and tie speaking into microphones

All BLIP descriptions:
1. a man in a suit and tie speaking into microphones
2. a man in a suit and tie giving a speech
3. a man in a suit and tie giving a speech

CSV text:
 if you think t willl stop eating palony and viannas voetsek!!! stupid no brain here

Analysis JSON:
 {'description': 'A man in a suit and tie is speaking at a podium with multiple microphones, appearing to address an audience or media.', 'keywords': ['man', 'suit', 'tie', 's